# Day 031 | Feature Engineering Techniques## 100 Days of ML for Geotechnical Engineering**Phase:** Data Science Essentials**Author:** Ripon Chandra Malo  **Date:** 2026---## What We Will Learn Today| Concept | What It Means | Geotechnical Example ||---------|--------------|---------------------|| **Polynomial Features** | Create squared/cubed terms | `depth²` captures nonlinear stress increase || **Interaction Features** | Combine two features | `N_value / depth` = normalized resistance || **Binning** | Group continuous values into ranges | Shallow / Medium / Deep layers || **Log Transform** | Compress skewed distributions | `log(N_value)` for heavy-tailed SPT data || **One-Hot Encoding** | Convert categories to binary columns | `soil_type` → `is_Clay`, `is_Sand`, ... || **Target Encoding** | Replace category with target mean | `Clay` → mean N-value for Clay |---## Our Geotechnical ProblemWe have raw borehole data from a site investigation: depth, N-value, soil type, water content, and other basic measurements. But raw features alone often miss important relationships.**Goal:** Transform raw soil data into a powerful ML-ready feature matrix. Good features are the difference between a mediocre model and an excellent one.> *"In geotechnical ML, feature engineering is where domain knowledge meets data science."*---

## Part 1: Load and Explore the Raw DataBefore engineering features, we need to understand what we have. Our dataset `raw_soil_data.csv` contains 80 borehole samples with basic measurements.

In [ ]:
# ─────────────────────────────────────────────────────# LOAD RAW BOREHOLE DATA# ─────────────────────────────────────────────────────import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport warningswarnings.filterwarnings('ignore')# ── Load the dataset ──df = pd.read_csv("raw_soil_data.csv")print("Raw Borehole Data")print("=" * 55)print(f"Samples : {len(df)}")print(f"Features: {df.shape[1]}")print(f"Columns : {list(df.columns)}")print()print(df.head(10))

In [ ]:
# ─────────────────────────────────────────────────────# QUICK STATISTICS# ─────────────────────────────────────────────────────print("Numerical Summary")print("─" * 55)print(df.describe().round(2))print()print("Soil Types Present:")print(df['soil_type'].value_counts())

## Part 2: Polynomial Features — Capturing Nonlinear RelationshipsIn geotechnical engineering, many relationships are nonlinear:- **Overburden stress** increases with depth² (quadratic)- **SPT resistance** often grows nonlinearly with depth- **Consolidation pressure** depends on depth squaredPolynomial features let our ML model capture these curves without us manually deriving each equation.---

In [ ]:
# ─────────────────────────────────────────────────────# POLYNOMIAL FEATURES: depth² and N_value²# ─────────────────────────────────────────────────────# Why depth²? Overburden stress σ'v increases with depth,# and many soil properties depend on effective stress.df['depth_squared'] = df['depth'] ** 2df['N_value_squared'] = df['N_value'] ** 2df['depth_cubed'] = df['depth'] ** 3print("Polynomial Features Created:")print("─" * 55)print(df[['depth', 'depth_squared', 'depth_cubed', 'N_value', 'N_value_squared']].head(8))print()print("Why this matters:")print("  depth=5.0  → depth²=25.0    (moderate stress)")print("  depth=20.0 → depth²=400.0   (high stress)")print("  The squared term amplifies deep samples, helping")print("  the model learn depth-dependent behavior.")

In [ ]:
# ─────────────────────────────────────────────────────# SKLEARN POLYNOMIAL FEATURES (automated approach)# ─────────────────────────────────────────────────────from sklearn.preprocessing import PolynomialFeatures# Select two features for polynomial expansionX_sample = df[['depth', 'N_value']].head(5)poly = PolynomialFeatures(degree=2, include_bias=False)X_poly = poly.fit_transform(X_sample)print("Original features: depth, N_value")print("After degree=2 polynomial expansion:")print(f"  Feature names: {poly.get_feature_names_out()}")print()print("Example (first 5 rows):")poly_df = pd.DataFrame(X_poly, columns=poly.get_feature_names_out())print(poly_df.round(1))print()print("Notice: sklearn automatically creates depth², N², AND depth×N!")

## Part 3: Interaction Features — Domain-Driven CombinationsInteraction features combine two or more raw features into a new one that captures a **physical relationship**. In geotechnical engineering, these are often well-known ratios:| Interaction Feature | Formula | Physical Meaning ||---|---|---|| Normalized resistance | `N / depth` | SPT blow count per meter of overburden || Depth × unit weight | `depth × γ` | Approximate overburden pressure || N × fines ratio | `N × (1 - FC/100)` | Resistance adjusted for fines |---

In [ ]:
# ─────────────────────────────────────────────────────# INTERACTION FEATURES# ─────────────────────────────────────────────────────# ── N / depth ratio (normalized penetration resistance) ──# Avoids division by zero with np.wheredf['N_over_depth'] = np.where(df['depth'] > 0,                                df['N_value'] / df['depth'],                                0)# ── Depth × unit_weight (approximate overburden stress) ──df['overburden_approx'] = df['depth'] * df['unit_weight']# ── N × depth product (captures resistance at depth) ──df['N_times_depth'] = df['N_value'] * df['depth']# ── Fines-adjusted N-value ──df['N_clean'] = df['N_value'] * (1 - df['fines_content'] / 100)print("Interaction Features:")print("─" * 55)print(df[['depth', 'N_value', 'N_over_depth', 'overburden_approx',            'N_clean']].head(8).round(2))print()print("Engineering insight:")print("  N/depth normalizes for overburden — a key concept in SPT analysis.")print("  This single feature can dramatically improve prediction accuracy.")

## Part 4: Binning — Converting Continuous to CategoricalSometimes it is useful to group continuous values into categories. In geotechnical engineering, we naturally think in ranges:- **Depth zones:** Shallow (0-5m), Medium (5-15m), Deep (>15m)- **N-value classes:** Very Loose, Loose, Medium, Dense, Very Dense- **Moisture levels:** Dry, Moist, Wet, SaturatedBinning captures these engineering categories that a linear model would miss.---

In [ ]:
# ─────────────────────────────────────────────────────# BINNING: Depth Zones# ─────────────────────────────────────────────────────# ── Manual binning with pd.cut ──depth_bins = [0, 5, 15, 30, float('inf')]depth_labels = ['Shallow', 'Medium', 'Deep', 'Very Deep']df['depth_zone'] = pd.cut(df['depth'], bins=depth_bins, labels=depth_labels)print("Depth Zone Distribution:")print("─" * 40)print(df['depth_zone'].value_counts().sort_index())print()# ── N-value classification (standard geotechnical practice) ──n_bins = [0, 4, 10, 30, 50, float('inf')]n_labels = ['Very Loose', 'Loose', 'Medium Dense', 'Dense', 'Very Dense']df['density_class'] = pd.cut(df['N_value'], bins=n_bins, labels=n_labels)print("Density Classification:")print("─" * 40)print(df['density_class'].value_counts().sort_index())

In [ ]:
# ─────────────────────────────────────────────────────# VISUALIZE BINNING RESULTS# ─────────────────────────────────────────────────────fig, axes = plt.subplots(1, 2, figsize=(12, 4))# Depth zonesdf['depth_zone'].value_counts().sort_index().plot(    kind='bar', ax=axes[0], color=['#2196F3', '#4CAF50', '#FF9800', '#F44336'],    edgecolor='black', alpha=0.8)axes[0].set_title('Samples by Depth Zone', fontweight='bold')axes[0].set_xlabel('Depth Zone')axes[0].set_ylabel('Count')axes[0].tick_params(axis='x', rotation=0)# Density classificationdf['density_class'].value_counts().sort_index().plot(    kind='bar', ax=axes[1], color=['#FFCDD2', '#FFE0B2', '#C8E6C9', '#81C784', '#2E7D32'],    edgecolor='black', alpha=0.8)axes[1].set_title('Samples by Density Class', fontweight='bold')axes[1].set_xlabel('Density Class')axes[1].set_ylabel('Count')axes[1].tick_params(axis='x', rotation=30)plt.tight_layout()plt.savefig('binning_results.png', dpi=100, bbox_inches='tight')plt.show()print("Binning converts continuous features into engineering categories.")

## Part 5: Log Transforms — Taming Skewed DistributionsMany geotechnical measurements are right-skewed:- N-values can range from 1 to 50+ (with a long tail)- Water content varies widely across soil types- Fines content often clusters near 0% or 100%Log transforms compress the high end and spread out the low end, making data more normally distributed — which many ML algorithms prefer.---

In [ ]:
# ─────────────────────────────────────────────────────# LOG TRANSFORMS# ─────────────────────────────────────────────────────# log1p = log(1 + x), safe for zero valuesdf['log_N_value'] = np.log1p(df['N_value'])df['log_depth'] = np.log1p(df['depth'])df['log_water_content'] = np.log1p(df['water_content'])# ── Compare distributions ──fig, axes = plt.subplots(2, 2, figsize=(10, 8))axes[0, 0].hist(df['N_value'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)axes[0, 0].set_title('N_value (Original)', fontweight='bold')axes[0, 0].set_xlabel('N_value')axes[0, 1].hist(df['log_N_value'], bins=20, color='coral', edgecolor='black', alpha=0.7)axes[0, 1].set_title('log(1 + N_value)', fontweight='bold')axes[0, 1].set_xlabel('log(1 + N_value)')axes[1, 0].hist(df['water_content'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)axes[1, 0].set_title('Water Content (Original)', fontweight='bold')axes[1, 0].set_xlabel('Water Content (%)')axes[1, 1].hist(df['log_water_content'], bins=20, color='coral', edgecolor='black', alpha=0.7)axes[1, 1].set_title('log(1 + Water Content)', fontweight='bold')axes[1, 1].set_xlabel('log(1 + WC)')plt.tight_layout()plt.savefig('log_transforms.png', dpi=100, bbox_inches='tight')plt.show()print("Skewness comparison:")print(f"  N_value:     original={df['N_value'].skew():.2f}  →  log={df['log_N_value'].skew():.2f}")print(f"  Water content: original={df['water_content'].skew():.2f}  →  log={df['log_water_content'].skew():.2f}")

## Part 6: One-Hot Encoding — Converting Soil Types to NumbersML models need **numbers**, not text. One-hot encoding converts each category into a binary (0/1) column:```soil_type = "Clay"  →  is_Clay=1, is_Sand=0, is_Silt=0, ...soil_type = "Sand"  →  is_Clay=0, is_Sand=1, is_Silt=0, ...```This is the standard way to handle categorical variables without imposing a false numerical order.---

In [ ]:
# ─────────────────────────────────────────────────────# ONE-HOT ENCODING# ─────────────────────────────────────────────────────# ── Method 1: pandas get_dummies ──soil_dummies = pd.get_dummies(df['soil_type'], prefix='soil')print("One-Hot Encoded Soil Types (first 8 rows):")print("─" * 55)print(soil_dummies.head(8))print()print(f"Original: 1 column (soil_type)")print(f"Encoded:  {soil_dummies.shape[1]} columns (one per soil type)")

In [ ]:
# ─────────────────────────────────────────────────────# ONE-HOT WITH SKLEARN (for pipelines)# ─────────────────────────────────────────────────────from sklearn.preprocessing import OneHotEncoderencoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')soil_encoded = encoder.fit_transform(df[['soil_type']])encoded_df = pd.DataFrame(    soil_encoded,     columns=encoder.get_feature_names_out(['soil_type']))print("sklearn OneHotEncoder output:")print(encoded_df.head(5))print()print(f"Categories found: {encoder.categories_[0]}")print()print("Tip: Use drop='first' to avoid multicollinearity")print("     (the dropped category is captured by all zeros)")

## Part 7: Target Encoding — Category MeansTarget encoding replaces each category with the **mean of the target variable** for that category. This is powerful when you have many categories.| Soil Type | Mean N-value | Encoded Value ||-----------|-------------|---------------|| Clay | ~12 | 12.0 || Sand | ~28 | 28.0 || Gravel | ~40 | 40.0 |**Warning:** Target encoding can cause data leakage if not done carefully. Always compute means on the training set only.---

In [ ]:
# ─────────────────────────────────────────────────────# TARGET ENCODING (manual implementation)# ─────────────────────────────────────────────────────# Using N_value as the "target" for demonstrationtarget_means = df.groupby('soil_type')['N_value'].mean()print("Mean N-value by Soil Type:")print("─" * 40)print(target_means.round(2).sort_values())print()# Apply target encodingdf['soil_target_encoded'] = df['soil_type'].map(target_means)print("Target Encoding Applied (first 10 rows):")print(df[['soil_type', 'N_value', 'soil_target_encoded']].head(10).to_string())print()print("IMPORTANT: In practice, compute means on TRAINING set only,")print("then apply those means to the test set to avoid data leakage.")

## Part 8: Complete Application — Raw to ML-ReadyLet us put it all together: take our raw borehole data and produce a clean, ML-ready feature matrix.---

In [ ]:
# ═════════════════════════════════════════════════════# COMPLETE FEATURE ENGINEERING PIPELINE# ═════════════════════════════════════════════════════# ── Start fresh from raw data ──raw = pd.read_csv("raw_soil_data.csv")print(f"Starting with {raw.shape[1]} raw features, {len(raw)} samples")print(f"Raw columns: {list(raw.columns)}")print()# ── Step 1: Polynomial features ──raw['depth_sq'] = raw['depth'] ** 2raw['N_sq'] = raw['N_value'] ** 2# ── Step 2: Interaction features ──raw['N_over_depth'] = raw['N_value'] / raw['depth']raw['overburden'] = raw['depth'] * raw['unit_weight']raw['N_clean'] = raw['N_value'] * (1 - raw['fines_content'] / 100)# ── Step 3: Log transforms ──raw['log_N'] = np.log1p(raw['N_value'])raw['log_depth'] = np.log1p(raw['depth'])# ── Step 4: Binning ──raw['depth_zone'] = pd.cut(raw['depth'], bins=[0, 5, 15, 30, 100],                            labels=['Shallow', 'Medium', 'Deep', 'VDeep'])# ── Step 5: One-hot encode soil_type ──soil_ohe = pd.get_dummies(raw['soil_type'], prefix='soil', dtype=int)# ── Step 6: One-hot encode depth_zone ──zone_ohe = pd.get_dummies(raw['depth_zone'], prefix='zone', dtype=int)# ── Combine into feature matrix ──numeric_cols = ['depth', 'N_value', 'water_content', 'unit_weight',                'fines_content', 'liquid_limit', 'plasticity_index',                'depth_sq', 'N_sq', 'N_over_depth', 'overburden',                'N_clean', 'log_N', 'log_depth']X = pd.concat([raw[numeric_cols], soil_ohe, zone_ohe], axis=1)print(f"ML-ready feature matrix: {X.shape[0]} samples x {X.shape[1]} features")print(f"Feature names: {list(X.columns)}")print()print(X.head(5).round(2))

In [ ]:
# ─────────────────────────────────────────────────────# FEATURE IMPORTANCE INTUITION# ─────────────────────────────────────────────────────# Let's see which features correlate most with N_valuecorrelations = X.corrwith(raw['N_value']).abs().sort_values(ascending=False)print("Feature Correlation with N_value (absolute):")print("─" * 50)for feat, corr in correlations.head(15).items():    bar = "█" * int(corr * 30)    print(f"  {feat:25s} {corr:.3f}  {bar}")print()print("Engineered features like N_sq and log_N appear near the top!")print("This shows feature engineering captures valuable information.")

---## Practice ExercisesTry these on your own to strengthen your feature engineering skills:### Exercise 1: Create a new interaction featureCreate `water_content / liquid_limit` (called the **liquidity index** approximation). What does it tell you about soil consistency?### Exercise 2: Custom binningCreate bins for `fines_content`: Granular (<15%), Mixed (15-50%), Fine-grained (>50%). Count samples in each bin.### Exercise 3: Square root transformApply `np.sqrt()` to N_value. Compare its skewness to the log transform. Which works better?---

In [ ]:
# ─────────────────────────────────────────────────────# EXERCISE 1: Liquidity index approximation# ─────────────────────────────────────────────────────# Hint: Only valid where liquid_limit > 0# df['liquidity_approx'] = np.where(df['liquid_limit'] > 0,#                                    df['water_content'] / df['liquid_limit'],#                                    np.nan)# print(df[['soil_type', 'water_content', 'liquid_limit', 'liquidity_approx']].head(10))

In [ ]:
# ─────────────────────────────────────────────────────# EXERCISE 2: Fines content binning# ─────────────────────────────────────────────────────# fc_bins = [0, 15, 50, 100]# fc_labels = ['Granular', 'Mixed', 'Fine-grained']# df['fc_class'] = pd.cut(df['fines_content'], bins=fc_bins, labels=fc_labels)# print(df['fc_class'].value_counts().sort_index())

In [ ]:
# ─────────────────────────────────────────────────────# EXERCISE 3: Square root transform# ─────────────────────────────────────────────────────# df['sqrt_N'] = np.sqrt(df['N_value'])# print(f"Original skew: {df['N_value'].skew():.3f}")# print(f"Log skew:      {np.log1p(df['N_value']).skew():.3f}")# print(f"Sqrt skew:     {df['sqrt_N'].skew():.3f}")

---## Key Takeaways1. **Feature engineering is where domain knowledge meets data science.** Knowing that overburden stress depends on depth² is geotechnical knowledge that creates powerful polynomial features.2. **Interaction features** like `N/depth` capture physical relationships that raw features alone cannot express. These often become the most important predictors.3. **Binning** translates continuous measurements into the engineering categories we naturally think in (Shallow/Medium/Deep, Loose/Dense).4. **Log transforms** tame skewed distributions, making data friendlier for algorithms that assume normality.5. **One-hot encoding** is essential for categorical variables. Target encoding is a powerful alternative when you have many categories.---## How This Connects to ML| What We Did Today | What It Becomes in ML ||---|---|| Polynomial features (depth²) | Captures nonlinear patterns without nonlinear models || Interaction features (N/depth) | Domain-informed predictors that boost accuracy || Binning (depth zones) | Decision tree friendly; captures threshold effects || Log transforms | Better convergence for gradient-based algorithms || One-hot encoding | Required input format for most classifiers || Target encoding | Efficient encoding for high-cardinality categories |---*Day 031 of 100 — Keep going!*